# Q22: BLEU Score
**Topic:** NLP | **Difficulty:** Medium | **Time:** 6 min
**Sheet:** Grind75ML

---

## Question
What does BLEU mean?

## Detailed Answer

**BLEU** = **B**ilingual **E**valuation **U**nderstudy

A metric for evaluating the quality of machine-generated text against reference translations.

### Formula

$$\text{BLEU} = BP \cdot \exp\left(\sum_{n=1}^{N} w_n \log p_n\right)$$

Where:
- $p_n$ = **modified n-gram precision** (fraction of candidate n-grams found in reference)
- $w_n = 1/N$ (uniform weights, typically N=4)
- $BP$ = **brevity penalty** to penalize short translations:

$$BP = \begin{cases} 1 & \text{if } c > r \\ e^{1-r/c} & \text{if } c \leq r \end{cases}$$

where $c$ = candidate length, $r$ = reference length.

### Modified Precision
Each n-gram is counted at most as many times as it appears in the reference (clipped count). This prevents gaming by repeating common words.

### Properties
| Property | Detail |
|----------|--------|
| Range | 0 to 1 (higher is better) |
| Perfect match | BLEU = 1.0 |
| Typical good score | 0.3-0.5 |
| Granularity | Corpus-level (not sentence-level) |

### Limitations
- No semantic understanding (synonyms don't help)
- Doesn't capture fluency or grammar
- Sentence-level BLEU is unstable
- Alternatives: **ROUGE** (recall-oriented), **BERTScore** (semantic), **METEOR** (synonyms + order)

In [ ]:
import numpy as np
from collections import Counter


def get_ngrams(tokens: list[str], n: int) -> list[tuple]:
    """Extract n-grams from token list."""
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]


def modified_precision(candidate: list[str], references: list[list[str]], n: int) -> float:
    """Compute modified n-gram precision."""
    cand_ngrams = Counter(get_ngrams(candidate, n))
    max_ref_counts = Counter()
    for ref in references:
        ref_ngrams = Counter(get_ngrams(ref, n))
        for ng in cand_ngrams:
            max_ref_counts[ng] = max(max_ref_counts[ng], ref_ngrams[ng])
    
    clipped = sum(min(cand_ngrams[ng], max_ref_counts[ng]) for ng in cand_ngrams)
    total = sum(cand_ngrams.values())
    return clipped / total if total > 0 else 0


def bleu_score(candidate: list[str], references: list[list[str]], max_n: int = 4) -> float:
    """Compute BLEU score."""
    # Modified precisions
    precisions = []
    for n in range(1, max_n + 1):
        p = modified_precision(candidate, references, n)
        precisions.append(p)
    
    # Brevity penalty
    c = len(candidate)
    r = min(len(ref) for ref in references)  # closest reference length
    bp = 1.0 if c > r else np.exp(1 - r / c) if c > 0 else 0
    
    # Geometric mean of precisions
    if any(p == 0 for p in precisions):
        return 0.0
    log_avg = sum(np.log(p) for p in precisions) / max_n
    
    return bp * np.exp(log_avg)


# --- Example ---
ref1 = 'the cat is on the mat'.split()
ref2 = 'there is a cat on the mat'.split()
cand1 = 'the cat sat on the mat'.split()  # good translation
cand2 = 'the the the the the the'.split()  # degenerate

print(f'Good candidate:  BLEU = {bleu_score(cand1, [ref1, ref2]):.4f}')
print(f'Bad candidate:   BLEU = {bleu_score(cand2, [ref1, ref2]):.4f}')

# Per-n-gram precision breakdown
for n in range(1, 5):
    p = modified_precision(cand1, [ref1, ref2], n)
    print(f'  {n}-gram precision: {p:.4f}')

## Key Takeaways
- BLEU measures **precision** of n-grams — how many predicted n-grams appear in the reference
- **Brevity penalty** prevents short, high-precision but low-recall translations
- Modified precision **clips counts** to prevent gaming
- For modern evaluation, prefer **BERTScore** or **COMET** which capture semantic similarity